# ReLU Tessellation Dynamics: Standard vs Adversarial Training

This notebook runs the full experimental pipeline on Google Colab with GPU support.

**Requirements:** Select a **GPU runtime** before running (Runtime > Change runtime type > T4 GPU).

The setup installs:
- `graph-tool` via conda (required by SplineCam for exact tessellation computation)
- SplineCam from the official repository
- All other Python dependencies (torch, numpy, scipy, matplotlib, etc.)

## 0. Verify GPU availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
        "then restart the runtime."
    )

## 1. Install graph-tool via conda (matched to kernel Python)

`graph-tool` is a C++ library not available via pip. The Ubuntu apt package is built for Python 3.10, but Colab's kernel runs Python 3.12 — the compiled extensions are ABI-incompatible across minor versions.

We install miniconda and use conda-forge to get a graph-tool build that matches the kernel's exact Python version. This takes 3-5 minutes.

In [ ]:
import sys
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"Colab kernel Python: {PY_VER} ({sys.executable})")
print(f"We will install graph-tool for Python {PY_VER} via conda-forge.")

## 2. Install miniconda, graph-tool, and fix system libraries

The next two cells handle:
1. **Cell A**: Installs miniconda + graph-tool via conda-forge (skip if already done)
2. **Cell B**: Copies the conda env's newer `libstdc++` over the system version and **restarts the runtime** (only on first run — it detects if already done)
3. **Cell C**: Adds the conda env to `sys.path` and imports `graph_tool`

After the restart, re-run all cells from the top. On second run, Cell B will skip the restart and Cell C will import normally.

In [ ]:
import os, sys

CONDA_PREFIX = "/opt/conda"
CONDA = f"{CONDA_PREFIX}/bin/conda"
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"  # Colab kernel version
GT_ENV = f"{CONDA_PREFIX}/envs/gt"

# Step 1: Install miniconda if not present
if not os.path.exists(CONDA):
    print("=== Installing miniconda ===")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p {CONDA_PREFIX} 2>&1 | tail -2
    !rm /tmp/miniconda.sh

# Step 2: Configure conda — remove defaults entirely, use only conda-forge
!{CONDA} config --remove channels defaults 2>/dev/null; true
!{CONDA} config --add channels conda-forge 2>/dev/null; true
!{CONDA} config --set channel_priority strict
print("Conda channels configured (conda-forge only)")

# Step 3: Create env with --override-channels to fully bypass any default/TOS channels
if not os.path.exists(GT_ENV):
    print(f"\n=== Creating conda env 'gt' with Python {PY_VER} + graph-tool ===")
    print("(This takes 3-5 minutes...)")
    !{CONDA} create -y --override-channels -c conda-forge -n gt python={PY_VER} graph-tool 2>&1 | tail -15
else:
    print(f"Conda env 'gt' already exists at {GT_ENV}")

# Step 4: Verify
print("\n=== Verification ===")
gt_python = f"{GT_ENV}/bin/python"
if os.path.exists(gt_python):
    !{gt_python} --version
    !{gt_python} -c "import graph_tool; print('graph-tool', graph_tool.__version__)"
else:
    print(f"ERROR: {gt_python} not found — env creation may have failed.")
    print("Check the output above for errors.")

In [ ]:
import os, sys, shutil, subprocess

GT_ENV = "/opt/conda/envs/gt"
conda_lib = f"{GT_ENV}/lib"
dst_dir = "/lib/x86_64-linux-gnu"
src = os.path.realpath(f"{conda_lib}/libstdc++.so.6")
dst = f"{dst_dir}/libstdc++.so.6"

# Check if the system libstdc++ already has CXXABI_1.3.15
result = subprocess.run(["strings", dst], capture_output=True, text=True)
if "CXXABI_1.3.15" in result.stdout:
    print("System libstdc++ already has CXXABI_1.3.15 — no restart needed.")
else:
    print(f"Copying {src} -> {dst}")
    shutil.copy2(src, dst)
    # Verify the copy worked
    result2 = subprocess.run(["strings", dst], capture_output=True, text=True)
    assert "CXXABI_1.3.15" in result2.stdout, "Copy failed — CXXABI_1.3.15 still missing"
    print("Library updated. Restarting runtime to load the new version...")
    print(">>> After restart, re-run from Cell 0 (GPU check) through here, then continue. <<<")
    os.kill(os.getpid(), 9)  # Force kernel restart

In [ ]:
import sys, os

PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
GT_ENV = "/opt/conda/envs/gt"
conda_sp = f"{GT_ENV}/lib/python{PY_VER}/site-packages"
conda_lib = f"{GT_ENV}/lib"

# Add conda env's site-packages and libraries to the search path
if conda_sp not in sys.path:
    sys.path.insert(0, conda_sp)
ld_path = os.environ.get("LD_LIBRARY_PATH", "")
if conda_lib not in ld_path:
    os.environ["LD_LIBRARY_PATH"] = f"{conda_lib}:{ld_path}"

print(f"Python: {PY_VER}")
print(f"conda site-packages: {conda_sp}")

import graph_tool
print(f"\ngraph-tool version: {graph_tool.__version__}")
print(f"Location: {graph_tool.__file__}")

# Install remaining pip packages into Colab's default Python
!pip install -q torch torchvision matplotlib numpy scipy seaborn tqdm \
    python-igraph networkx livelossplot

import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## 3. Clone repositories and install SplineCam

In [ ]:
import os, sys, re

# Clone SplineCam
SPLINECAM_DIR = "/content/splinecam"
if not os.path.exists(SPLINECAM_DIR):
    !git clone https://github.com/AhmedImtiazPrio/splinecam.git {SPLINECAM_DIR}
else:
    print(f"SplineCam already cloned at {SPLINECAM_DIR}")

# --- Patch SplineCam for Python 3.12 + modern PyTorch compatibility ---

def annotate_def_params(line):
    """Add type annotations to unannotated default params in a def line."""
    match = re.match(r'(.*def \w+\()(.*)(\):.*)', line)
    if not match:
        return line
    prefix, params_str, suffix = match.groups()
    new_params = []
    for param in params_str.split(','):
        param = param.strip()
        if not param or '=' not in param:
            new_params.append(param)
            continue
        name_part, val_part = param.split('=', 1)
        name_part = name_part.strip()
        val_part = val_part.strip()
        if ':' in name_part:
            new_params.append(f"{name_part}={val_part}")
            continue
        if re.match(r'^-?\d+e[+-]?\d+$', val_part) or re.match(r'^-?\d+\.\d+$', val_part):
            new_params.append(f"{name_part}: float={val_part}")
        elif re.match(r'^-?\d+$', val_part):
            if 'pad_dist' in name_part:
                new_params.append(f"{name_part}: float={val_part}.0")
            else:
                new_params.append(f"{name_part}: int={val_part}")
        else:
            new_params.append(f"{name_part}={val_part}")
    return prefix + ', '.join(new_params) + suffix + '\n'

# ── Patch 1: utils.py — TorchScript type annotations ──
utils_path = os.path.join(SPLINECAM_DIR, "splinecam", "utils.py")
with open(utils_path, "r") as f:
    lines = f.readlines()
original = ''.join(lines)
patched_lines = []
for line in lines:
    if line.strip().startswith("def ") and "=" in line:
        line = annotate_def_params(line)
    patched_lines.append(line)
patched = ''.join(patched_lines)
if patched != original:
    with open(utils_path, "w") as f:
        f.write(patched)
    print("Patched utils.py: TorchScript type annotations")
else:
    print("utils.py: already patched")

# ── Patch 2: graph.py — escape sequence + len() on 0-D tensors ──
graph_path = os.path.join(SPLINECAM_DIR, "splinecam", "graph.py")
with open(graph_path, "r") as f:
    content = f.read()
orig_graph = content

# Fix invalid escape sequence \i (Python 3.12 SyntaxWarning)
if "\\i" in content and "\\\\i" not in content:
    content = content.replace("\\i", "\\\\i")

# Fix len() calls on tensors that may be 0-dimensional.
# Instead of a try/except wrapper (TorchScript doesn't support try),
# directly replace len(var) with var.shape[0] for tensor variables.
# .shape[0] works on 1-D+ tensors and raises IndexError (not TypeError)
# on 0-D tensors, but the real fix is to ensure these are always 1-D
# by reshaping the torch.where output. We use .reshape(-1) approach:
# Replace the torch.where patterns to always produce 1-D output.
if '_patched_len' not in content:
    # Replace specific len() calls with .shape[0] which is safer
    # But better: wrap torch.where results with .reshape(-1) to guarantee 1-D
    content = content.replace(
        'no_inter_idx = torch.where(torch.prod(q[:-1] == q[1:],axis=1))[0].cpu()',
        'no_inter_idx = torch.where(torch.prod(q[:-1] == q[1:],axis=1))[0].reshape(-1).cpu()'
    )
    content = content.replace(
        'inter_idx = torch.where(~torch.prod(q[:-1] == q[1:],axis=1))[0].cpu()',
        'inter_idx = torch.where(~torch.prod(q[:-1] == q[1:],axis=1))[0].reshape(-1).cpu()'
    )
    # Also fix any other torch.where()[0] patterns to add .reshape(-1)
    content = re.sub(
        r'torch\.where\(([^)]+)\)\[0\](?!\.reshape)',
        r'torch.where(\1)[0].reshape(-1)',
        content
    )
    # Mark as patched
    content = content.replace(
        'import torch\n',
        'import torch\n_patched_len = True\n',
        1
    )

if content != orig_graph:
    with open(graph_path, "w") as f:
        f.write(content)
    print("Patched graph.py: escape sequence + reshape(-1) for torch.where outputs")
else:
    print("graph.py: already patched")

# ── Patch 3: compute.py — disable multiprocessing ──
compute_path = os.path.join(SPLINECAM_DIR, "splinecam", "compute.py")
with open(compute_path, "r") as f:
    content = f.read()
if "num_workers=n_workers" in content:
    content = content.replace("num_workers=n_workers", "num_workers=0")
    with open(compute_path, "w") as f:
        f.write(content)
    print("Patched compute.py: set DataLoader num_workers=0")

# Add to sys.path
if SPLINECAM_DIR not in sys.path:
    sys.path.insert(0, SPLINECAM_DIR)

# Install SplineCam's dependencies
!pip install -q python-igraph networkx

# Verify
import splinecam
print(f"\nSplineCam imported successfully from {splinecam.__file__}")

In [ ]:
import os

PROJECT_DIR = "/content/adversarial-tesselation-analysis"
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/BryanZhang938/adversarial-tesselation-analysis.git {PROJECT_DIR}
else:
    print(f"Project already cloned at {PROJECT_DIR}")

assert os.path.exists(PROJECT_DIR), f"Project not found at {PROJECT_DIR}"
print(f"Project directory: {PROJECT_DIR}")
print(os.listdir(PROJECT_DIR))

# ── Patch tessellation_analysis.py: fix SplineCam API call ──
# The function compute_splinecam_tessellation needs to pass 3 separate args
# to get_partitions_with_db(domain_tensor, T_projection, NN_wrapper)
tess_path = os.path.join(PROJECT_DIR, "src", "tessellation_analysis.py")
with open(tess_path, "r") as f:
    content = f.read()

needs_patch = False

# Fix 1: model_wrapper must use as_sequential=False
if "as_sequential=False" not in content and "model_wrapper(" in content:
    content = content.replace(
        "device=device,\n    )",
        "device=device,\n        as_sequential=False,\n    )"
    )
    needs_patch = True

# Fix 2: get_partitions_with_db needs (domain_tensor, T_proj, NN) not (domain, T, None)
if "regions, db_edges = splinecam.compute.get_partitions_with_db(" in content:
    old_call = '''    # Compute partitions with decision boundary
    # Third arg is projection matrix (None for 2D)
    regions, db_edges = splinecam.compute.get_partitions_with_db(
        domain, T, None
    )

    return regions, db_edges, T'''
    new_call = '''    # get_partitions_with_db(domain, T, NN) expects:
    #   domain: (V, 2) tensor of domain vertices on GPU
    #   T: (2, 3) projection matrix [I | 0] for 2D inputs
    #   NN: the model_wrapper object
    domain_t = torch.from_numpy(domain).to(device)
    T_proj = torch.eye(3, dtype=torch.float64, device=device)[:2]  # 2x3: [I | 0]

    regions, db_edges, Abw = splinecam.compute.get_partitions_with_db(
        domain_t, T_proj, NN,
        fwd_batch_size=512, batch_size=64, n_workers=0
    )

    return regions, db_edges, NN'''
    if old_call in content:
        content = content.replace(old_call, new_call)
        needs_patch = True

# Fix 3: Rename T variable to NN in model_wrapper call
if "T = splinecam.wrappers.model_wrapper(" in content:
    content = content.replace(
        "T = splinecam.wrappers.model_wrapper(",
        "NN = splinecam.wrappers.model_wrapper("
    )
    needs_patch = True

if needs_patch:
    with open(tess_path, "w") as f:
        f.write(content)
    print("Patched tessellation_analysis.py: fixed SplineCam API call")
else:
    print("tessellation_analysis.py: already correct")

## 4. Patch SplineCam for Colab compatibility

SplineCam has two known issues in Colab:
1. **TorchScript float typing**: `utils.py` uses integer defaults (e.g. `pad_dist=1`) which TorchScript rejects. We patch them to floats.
2. **Multiprocessing + CUDA**: `compute.py` spawns workers that try to initialize CUDA, causing errors. We patch DataLoader `num_workers` to 0.

In [ ]:
import importlib
import torch

# ── Monkey-patch torch.Tensor.__len__ to handle 0-D tensors ──
# SplineCam calls len() on tensors from torch.where() which can be 0-D.
# Standard PyTorch raises "len() of unsized object" for 0-D tensors.
# This patch makes 0-D tensors return 1 (they contain one element).
_orig_tensor_len = torch.Tensor.__len__
def _patched_tensor_len(self):
    if self.dim() == 0:
        return 1
    return _orig_tensor_len(self)
torch.Tensor.__len__ = _patched_tensor_len
print("Patched torch.Tensor.__len__ to handle 0-D tensors")

# Reload SplineCam modules to pick up file patches
import splinecam.utils
import splinecam.compute
importlib.reload(splinecam.utils)
importlib.reload(splinecam.compute)
# Note: graph.py is NOT reloaded here since it triggers TorchScript
# compilation errors. The file-level patches are picked up on first import.
print("SplineCam modules reloaded with all patches applied.")

## 5. Verify SplineCam works end-to-end

Quick smoke test: wrap a tiny MLP and compute partitions on a small domain.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from splinecam.wrappers import model_wrapper
from splinecam.compute import get_partitions_with_db

# Build a tiny test model
test_model = nn.Sequential(
    nn.Linear(2, 10),
    nn.ReLU(),
    nn.Linear(10, 2),
).eval().cuda()

# Define square domain
domain = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=np.float64)
domain_t = torch.from_numpy(domain).cuda()

# Wrap model — as_sequential=False keeps layers as a list,
# which is required because get_partitions_with_db uses slicing (NN.layers[:n])
NN = model_wrapper(test_model, input_shape=(2,), T=None,
                   dtype=torch.float64, device='cuda',
                   as_sequential=False)

# Compute tessellation
T_proj = torch.eye(3, dtype=torch.float64, device='cuda')[:2]  # 2x3: [I | 0]
regions, endpoints, Abw = get_partitions_with_db(
    domain_t, T_proj, NN,
    fwd_batch_size=512, batch_size=64, n_workers=0
)

print(f"Tessellation computed successfully!")
print(f"  Number of regions: {len(regions)}")
print(f"  Endpoints type: {type(endpoints)}")
del test_model, NN, regions, endpoints, Abw
torch.cuda.empty_cache()

## 6. (Gap, ε) Feasibility Sweep — Connecting Tessellation Geometry to Shafahi et al. Bounds

**Key theoretical result (Shafahi et al., 2018):** For two classes separated by a gap $\Delta$, robust classification at perturbation budget $\varepsilon$ is:
- **Feasible** when $\Delta > 2\varepsilon$ (there exists a classifier with nonzero robust accuracy)
- **Infeasible** when $\Delta < 2\varepsilon$ (no classifier can be robust)

**Experiment:** We use concentric rings with controllable gap to sweep over (gap, ε) combinations and show that the tessellation phenomena from adversarial training (fewer regions, larger cells, greater boundary distance) hold in the feasible regime and break down in the infeasible regime.

In [ ]:
import sys, os, importlib, copy, json, time
import numpy as np
import torch
import torch.nn as nn

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

PROJECT_DIR = "/content/adversarial-tesselation-analysis"
sys.path.insert(0, PROJECT_DIR)

# Reload all project modules to pick up changes
import src.datasets; importlib.reload(src.datasets)
import src.models; importlib.reload(src.models)
import src.adversarial; importlib.reload(src.adversarial)
import src.train; importlib.reload(src.train)
import src.tessellation_analysis; importlib.reload(src.tessellation_analysis)
import src.visualization; importlib.reload(src.visualization)
import configs.experiment_config; importlib.reload(configs.experiment_config)

from src.datasets import make_concentric_rings_gap, get_dataloader
from src.models import make_relu_mlp, count_parameters
from src.train import train_model, evaluate_accuracy, evaluate_robust_accuracy
from src.tessellation_analysis import analyze_checkpoint, compute_grid_statistics
from src.visualization import plot_dataset, plot_decision_regions, plot_tessellation_regions
from configs.experiment_config import make_config, _compute_checkpoint_epochs

from torch.utils.data import TensorDataset

print("All modules loaded successfully.")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# =====================================================
# EXPERIMENT CONFIGURATION — (gap, epsilon) sweep
# =====================================================
# Pre-normalization gap values.  After normalization the effective gap
# shrinks by a dataset-dependent scale factor; we record both.
GAP_VALUES = [0.1, 0.2, 0.3, 0.5, 0.8]

# Epsilon values (post-normalization, in data coordinates)
# We will compare each effective_gap to 2*eps to classify feasibility.
EPS_VALUES = [0.02, 0.05, 0.10, 0.15, 0.25]

# Shared training hyper-parameters (matched architecture for fair comparison)
HIDDEN_DIMS = [200]        # single wide hidden layer
EPOCHS      = 500          # enough for convergence on rings
LR          = 0.1          # higher LR + cosine annealing
SCHEDULER   = "cosine"
OPTIMIZER   = "sgd"
BATCH_SIZE  = 64
PGD_STEPS   = 7
NORM        = "l2"
N_SAMPLES   = 500
NOISE       = 0.02         # << gap/2 to keep classes well-separated
SEED        = 42
NUM_CKPTS   = 10           # fewer checkpoints — we mainly care about final state
RESOLUTION  = 150          # grid resolution (lower for speed in sweep)

print(f"Sweep: {len(GAP_VALUES)} gaps × {len(EPS_VALUES)} epsilons "
      f"= {len(GAP_VALUES)*len(EPS_VALUES)} (gap, eps) pairs")
print(f"Architecture: {HIDDEN_DIMS}, {EPOCHS} epochs, lr={LR}, {SCHEDULER}")
print(f"Each pair trains 2 models (ERM + PGD-AT) → "
      f"{2*len(GAP_VALUES)*len(EPS_VALUES)} total training runs")

## 7. Visualize datasets at each gap value

In [ ]:
fig, axes = plt.subplots(1, len(GAP_VALUES), figsize=(4*len(GAP_VALUES), 4))
for i, gap in enumerate(GAP_VALUES):
    X, y, eff_gap = make_concentric_rings_gap(
        n_samples=N_SAMPLES, gap=gap, noise=NOISE, seed=SEED
    )
    ax = axes[i]
    plot_dataset(X, y, title=f"gap={gap:.1f}\neff={eff_gap:.3f}", ax=ax)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
plt.suptitle("Concentric Rings at Different Gap Values", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Run the (gap, ε) sweep

For each (gap, ε) pair, train both an ERM model and a PGD-AT model with matched architecture. Record final-epoch tessellation statistics and convergence metrics.

In [ ]:
def run_single_training(gap, eps, adv_enabled, hidden_dims, epochs, lr,
                        scheduler, seed, device):
    """Train one model for a given (gap, eps) pair and return final stats."""
    # Generate dataset
    X_train, y_train, eff_gap = make_concentric_rings_gap(
        n_samples=N_SAMPLES, gap=gap, noise=NOISE, seed=seed
    )
    X_test, y_test, _ = make_concentric_rings_gap(
        n_samples=200, gap=gap, noise=NOISE, seed=seed + 1000
    )

    X_tensor = torch.from_numpy(X_train)
    y_tensor = torch.from_numpy(y_train)
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = get_dataloader(dataset, batch_size=BATCH_SIZE)

    # Build config
    cfg = make_config(
        dataset="concentric_rings",
        hidden_dims=hidden_dims,
        epochs=epochs,
        lr=lr,
        optimizer=OPTIMIZER,
        scheduler=scheduler,
        adv_enabled=adv_enabled,
        epsilon=eps,
        pgd_steps=PGD_STEPS,
        norm=NORM,
        num_checkpoints=NUM_CKPTS,
        resolution=RESOLUTION,
        noise=NOISE,
        n_samples=N_SAMPLES,
        seed=seed,
        device=device,
    )

    # Build & train model
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = make_relu_mlp(input_dim=2, hidden_dims=hidden_dims, output_dim=2)

    run_name = f"gap{gap:.2f}_eps{eps:.3f}_{'adv' if adv_enabled else 'std'}"
    ckpt_dir = os.path.join("/content/checkpoints_sweep", run_name)

    history = train_model(
        model, dataloader, cfg,
        checkpoint_dir=ckpt_dir,
        run_name=run_name,
        device=device,
        X_test=X_test,
        y_test=y_test,
    )

    # Compute final tessellation statistics
    model.eval()
    stats, grid_data = compute_grid_statistics(
        model,
        domain_range=(-1.5, 1.5),
        resolution=RESOLUTION,
        data_points=X_train,
        device=device,
        local_complexity_radius=0.1,
    )

    # Final accuracy
    final_clean_acc = evaluate_accuracy(model, X_test, y_test, device=device)
    final_robust_acc = 0.0
    if adv_enabled:
        final_robust_acc = evaluate_robust_accuracy(
            model, X_test, y_test, epsilon=eps,
            num_steps=PGD_STEPS, norm=NORM, device=device
        )

    result = {
        "gap": gap,
        "eps": eps,
        "effective_gap": eff_gap,
        "feasible": eff_gap > 2 * eps,
        "adv_enabled": adv_enabled,
        "final_clean_acc": final_clean_acc,
        "final_robust_acc": final_robust_acc,
        "num_regions": stats["num_regions_grid"],
        "local_complexity": stats.get("local_region_count_mean", 0),
        "boundary_density": stats.get("boundary_density", 0),
        "boundary_dist_mean": stats.get("boundary_dist_mean", 0),
        "cell_area_median": stats.get("cell_area_median", 0),
        "decision_boundary_density": stats.get("decision_boundary_density", 0),
    }
    return result, history, stats, grid_data, X_train, y_train

print("Training function defined.")

In [ ]:
# =====================================================
# RUN THE FULL SWEEP
# =====================================================
all_results = []       # flat list of result dicts
all_grids = {}         # keyed by (gap, eps, 'std'/'adv')
all_histories = {}

total = len(GAP_VALUES) * len(EPS_VALUES) * 2
run_idx = 0

for gap in GAP_VALUES:
    for eps in EPS_VALUES:
        for adv_enabled in [False, True]:
            run_idx += 1
            tag = "ADV" if adv_enabled else "STD"
            print(f"\n{'='*60}")
            print(f"[{run_idx}/{total}] gap={gap:.2f}, eps={eps:.3f}, mode={tag}")
            print(f"{'='*60}")

            t0 = time.time()
            result, hist, stats, gd, X_tr, y_tr = run_single_training(
                gap=gap, eps=eps, adv_enabled=adv_enabled,
                hidden_dims=HIDDEN_DIMS, epochs=EPOCHS, lr=LR,
                scheduler=SCHEDULER, seed=SEED, device=device,
            )
            elapsed = time.time() - t0

            key = (gap, eps, "adv" if adv_enabled else "std")
            all_results.append(result)
            all_grids[key] = (gd, X_tr, y_tr)
            all_histories[key] = hist

            feas = "FEASIBLE" if result["feasible"] else "INFEASIBLE"
            print(f"  {feas} (eff_gap={result['effective_gap']:.4f}, 2ε={2*eps:.4f})")
            print(f"  Clean acc: {result['final_clean_acc']:.4f} | "
                  f"Robust acc: {result['final_robust_acc']:.4f}")
            print(f"  Regions: {result['num_regions']} | "
                  f"Local complexity: {result['local_complexity']:.1f} | "
                  f"Boundary dist: {result['boundary_dist_mean']:.4f}")
            print(f"  Time: {elapsed:.1f}s")

print(f"\n{'='*60}")
print(f"Sweep complete: {len(all_results)} results collected.")
print(f"{'='*60}")

In [ ]:
# Build summary table
import pandas as pd

df = pd.DataFrame(all_results)
df["gap_over_2eps"] = df["effective_gap"] / (2 * df["eps"])
df["regime"] = df["feasible"].map({True: "feasible", False: "infeasible"})
df["mode"] = df["adv_enabled"].map({True: "PGD-AT", False: "ERM"})

# Pivot for readability
print("=== Summary Table ===")
print(df[["gap", "eps", "effective_gap", "regime", "mode",
          "final_clean_acc", "final_robust_acc",
          "num_regions", "local_complexity", "boundary_dist_mean"]].to_string(index=False))

df.head()

In [ ]:
# Convergence verification: plot final clean accuracy across the sweep
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, mode_label, mode_flag in [(axes[0], "ERM (Standard)", False),
                                   (axes[1], "PGD-AT (Adversarial)", True)]:
    sub = df[df["adv_enabled"] == mode_flag]
    for gap in GAP_VALUES:
        row = sub[sub["gap"] == gap]
        ax.plot(row["eps"], row["final_clean_acc"], 'o-',
                label=f"gap={gap:.1f}")
    ax.set_xlabel("ε (perturbation budget)")
    ax.set_ylabel("Final Test Clean Accuracy")
    ax.set_title(mode_label)
    ax.legend(fontsize=8)
    ax.set_ylim(0.4, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle("Convergence Check: Clean Accuracy Across (gap, ε) Sweep", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Feasibility Heatmaps — Tessellation Metrics Across (gap, ε) Space

The key visualization: 2D heatmaps showing how tessellation metrics differ between ERM and PGD-AT models across the (gap, ε) space. The theoretical feasibility boundary $\Delta_{\text{eff}} = 2\varepsilon$ is overlaid as a dashed line.

In [ ]:
# Compute the difference (ADV - STD) for each tessellation metric at each (gap, eps)
# and plot as heatmaps with the feasibility boundary overlaid.

def build_metric_grids(df, metric):
    """Build 2D arrays for ERM, ADV, and their difference for a given metric."""
    n_gaps = len(GAP_VALUES)
    n_eps = len(EPS_VALUES)
    grid_std = np.full((n_gaps, n_eps), np.nan)
    grid_adv = np.full((n_gaps, n_eps), np.nan)

    for i, gap in enumerate(GAP_VALUES):
        for j, eps in enumerate(EPS_VALUES):
            row_std = df[(df["gap"] == gap) & (df["eps"] == eps) & (~df["adv_enabled"])]
            row_adv = df[(df["gap"] == gap) & (df["eps"] == eps) & (df["adv_enabled"])]
            if len(row_std) > 0:
                grid_std[i, j] = row_std[metric].values[0]
            if len(row_adv) > 0:
                grid_adv[i, j] = row_adv[metric].values[0]

    return grid_std, grid_adv, grid_adv - grid_std


def get_feasibility_boundary_line(df):
    """Return (eps_line, gap_line) for the effective_gap = 2*eps boundary."""
    # For each gap value, find the effective gap (same across eps for same gap)
    eff_gaps = {}
    for gap in GAP_VALUES:
        sub = df[df["gap"] == gap]
        if len(sub) > 0:
            eff_gaps[gap] = sub["effective_gap"].values[0]
    return eff_gaps


eff_gaps = get_feasibility_boundary_line(df)

# Metrics to visualize
metrics = [
    ("num_regions", "# Linear Regions", "RdBu_r"),
    ("local_complexity", "Local Complexity", "RdBu_r"),
    ("boundary_dist_mean", "Mean Boundary Distance", "RdBu"),
    ("boundary_density", "Boundary Density", "RdBu_r"),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for col, (metric, label, cmap) in enumerate(metrics):
    grid_std, grid_adv, grid_diff = build_metric_grids(df, metric)

    # Row 0: Ratio (ADV / STD) — shows relative effect of adversarial training
    # Use log ratio to handle different scales across gaps
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(grid_std > 0, grid_adv / grid_std, 1.0)

    ax = axes[0, col]
    im = ax.imshow(ratio, origin='lower', aspect='auto', cmap=cmap,
                   extent=[-0.5, len(EPS_VALUES)-0.5, -0.5, len(GAP_VALUES)-0.5])
    ax.set_xticks(range(len(EPS_VALUES)))
    ax.set_xticklabels([f"{e:.2f}" for e in EPS_VALUES], fontsize=8)
    ax.set_yticks(range(len(GAP_VALUES)))
    ax.set_yticklabels([f"{g:.1f}" for g in GAP_VALUES], fontsize=8)
    ax.set_xlabel("ε")
    ax.set_ylabel("Gap (pre-norm)")
    ax.set_title(f"ADV/STD Ratio\n{label}", fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Mark feasibility boundary: for each gap, find the eps where eff_gap ≈ 2*eps
    for i, gap in enumerate(GAP_VALUES):
        eg = eff_gaps[gap]
        boundary_eps = eg / 2
        # Find position in EPS_VALUES
        for j in range(len(EPS_VALUES) - 1):
            if EPS_VALUES[j] <= boundary_eps <= EPS_VALUES[j+1]:
                frac = (boundary_eps - EPS_VALUES[j]) / (EPS_VALUES[j+1] - EPS_VALUES[j])
                ax.plot(j + frac, i, 'k*', markersize=12)
                break
        elif boundary_eps > EPS_VALUES[-1]:
            ax.plot(len(EPS_VALUES) - 0.5, i, 'k>', markersize=8)
        elif boundary_eps < EPS_VALUES[0]:
            ax.plot(-0.3, i, 'k<', markersize=8)

    # Mark feasible vs infeasible cells
    for i, gap in enumerate(GAP_VALUES):
        for j, eps in enumerate(EPS_VALUES):
            eg = eff_gaps[gap]
            if eg <= 2 * eps:
                ax.plot(j, i, 'rx', markersize=10, markeredgewidth=2)

    # Row 1: Absolute values (STD solid, ADV dashed) per gap
    ax2 = axes[1, col]
    for i, gap in enumerate(GAP_VALUES):
        color = plt.cm.viridis(i / (len(GAP_VALUES) - 1))
        ax2.plot(EPS_VALUES, grid_std[i, :], 'o-', color=color,
                 label=f"STD gap={gap:.1f}", markersize=4)
        ax2.plot(EPS_VALUES, grid_adv[i, :], 's--', color=color,
                 label=f"ADV gap={gap:.1f}", markersize=4, alpha=0.7)
    ax2.set_xlabel("ε")
    ax2.set_ylabel(label)
    ax2.set_title(f"{label}\n(solid=ERM, dashed=PGD-AT)", fontsize=10)
    ax2.grid(True, alpha=0.3)
    if col == 0:
        ax2.legend(fontsize=6, ncol=2, loc='upper left')

fig.suptitle("Tessellation Metrics Across (gap, ε) Space\n"
             "Red × = infeasible (eff_gap < 2ε)   |   Black ★ = feasibility boundary",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/content/feasibility_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved feasibility_heatmaps.png")

In [ ]:
# =====================================================
# FIGURE 2: Key scatter plot — gap/2ε ratio vs tessellation metric differences
# =====================================================
# This is the single most important figure: it shows that the tessellation
# reorganization correlates with the feasibility ratio gap/(2ε).

df_std = df[~df["adv_enabled"]].copy().reset_index(drop=True)
df_adv = df[df["adv_enabled"]].copy().reset_index(drop=True)

# Merge on (gap, eps) to compute per-pair differences
merged = df_std.merge(df_adv, on=["gap", "eps"], suffixes=("_std", "_adv"))
merged["gap_ratio"] = merged["effective_gap_std"] / (2 * merged["eps"])

# Compute relative differences: (ADV - STD) / STD
for metric in ["num_regions", "local_complexity", "boundary_dist_mean", "boundary_density"]:
    with np.errstate(divide='ignore', invalid='ignore'):
        merged[f"rel_diff_{metric}"] = np.where(
            merged[f"{metric}_std"] > 0,
            (merged[f"{metric}_adv"] - merged[f"{metric}_std"]) / merged[f"{metric}_std"],
            0.0
        )

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

plot_configs = [
    ("rel_diff_num_regions", "Δ Region Count (ADV−STD)/STD", axes[0, 0]),
    ("rel_diff_local_complexity", "Δ Local Complexity (ADV−STD)/STD", axes[0, 1]),
    ("rel_diff_boundary_dist_mean", "Δ Boundary Distance (ADV−STD)/STD", axes[1, 0]),
    ("rel_diff_boundary_density", "Δ Boundary Density (ADV−STD)/STD", axes[1, 1]),
]

for metric, ylabel, ax in plot_configs:
    feasible = merged[merged["feasible_std"]]
    infeasible = merged[~merged["feasible_std"]]

    ax.scatter(feasible["gap_ratio"], feasible[metric],
               c='#2196F3', s=80, label="Feasible (gap > 2ε)",
               edgecolors='black', linewidth=0.5, zorder=5)
    ax.scatter(infeasible["gap_ratio"], infeasible[metric],
               c='#FF5722', s=80, marker='X', label="Infeasible (gap < 2ε)",
               edgecolors='black', linewidth=0.5, zorder=5)

    ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2,
               label="Feasibility boundary\n(gap = 2ε)", alpha=0.7)
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.set_xlabel("gap / (2ε)   [> 1 = feasible]", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Tessellation Reorganization vs. Feasibility Ratio\n"
             "Shafahi et al. (2018): robust classification requires gap > 2ε",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/content/feasibility_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved feasibility_scatter.png")

In [ ]:
# =====================================================
# FIGURE 3: Robust accuracy vs feasibility ratio
# =====================================================
# Verify that PGD-AT actually achieves robust accuracy only in feasible regime

fig, ax = plt.subplots(figsize=(8, 5))

adv_only = df[df["adv_enabled"]].copy()
adv_only["gap_ratio"] = adv_only["effective_gap"] / (2 * adv_only["eps"])

feasible = adv_only[adv_only["feasible"]]
infeasible = adv_only[~adv_only["feasible"]]

ax.scatter(feasible["gap_ratio"], feasible["final_robust_acc"],
           c='#2196F3', s=100, label="Feasible (gap > 2ε)",
           edgecolors='black', linewidth=0.5, zorder=5)
ax.scatter(infeasible["gap_ratio"], infeasible["final_robust_acc"],
           c='#FF5722', s=100, marker='X', label="Infeasible (gap < 2ε)",
           edgecolors='black', linewidth=0.5, zorder=5)

ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, alpha=0.7,
           label="Feasibility boundary (gap = 2ε)")
ax.set_xlabel("gap / (2ε)", fontsize=12)
ax.set_ylabel("PGD-AT Robust Test Accuracy", fontsize=12)
ax.set_title("Robust Accuracy Confirms Shafahi et al. Feasibility Bound", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig("/content/robust_acc_vs_feasibility.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved robust_acc_vs_feasibility.png")

## 10. Decision Region Snapshots — Feasible vs Infeasible Cases

Select representative (gap, ε) pairs from each regime and visualize the tessellation side-by-side.

In [ ]:
from IPython.display import Image, display

# Select representative cases:
# 1. Strongly feasible: large gap, small eps (gap=0.8, eps=0.02)
# 2. Marginally feasible: gap ≈ 2eps (gap=0.3, eps=0.05 or similar)
# 3. Infeasible: small gap, large eps (gap=0.1, eps=0.25)

cases = []
for gap, eps, label in [
    (0.8, 0.02, "Strongly Feasible"),
    (0.3, 0.10, "Near Boundary"),
    (0.1, 0.25, "Infeasible"),
]:
    # Check if these exist in our results
    match = df[(df["gap"] == gap) & (df["eps"] == eps)]
    if len(match) > 0:
        eff_gap = match["effective_gap"].values[0]
        ratio = eff_gap / (2 * eps)
        cases.append((gap, eps, label, ratio))
        print(f"{label}: gap={gap}, eps={eps}, eff_gap={eff_gap:.4f}, "
              f"gap/(2eps)={ratio:.2f}")

fig, axes = plt.subplots(len(cases), 4, figsize=(20, 5*len(cases)))
if len(cases) == 1:
    axes = axes.reshape(1, -1)

for row, (gap, eps, label, ratio) in enumerate(cases):
    for col, (mode, mode_label) in enumerate([("std", "ERM"), ("adv", "PGD-AT")]):
        key = (gap, eps, mode)
        if key in all_grids:
            gd, X_tr, y_tr = all_grids[key]
            # Decision regions
            plot_decision_regions(gd, X_tr, y_tr,
                                 title=f"{label}\n{mode_label} | gap={gap}, ε={eps}",
                                 ax=axes[row, col*2])
            # Tessellation regions
            plot_tessellation_regions(gd, X_tr, y_tr,
                                     title=f"{mode_label} Tessellation\ngap/(2ε)={ratio:.2f}",
                                     ax=axes[row, col*2+1])

for row in range(len(cases)):
    axes[row, 0].set_ylabel(cases[row][2], fontsize=12, fontweight='bold')

fig.suptitle("Decision Regions & Tessellation: Feasible vs. Infeasible Regimes",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("/content/regime_snapshots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved regime_snapshots.png")

In [ ]:
# =====================================================
# FIGURE 5: Boundary distance distributions — feasible vs infeasible
# =====================================================
fig, axes = plt.subplots(1, len(cases), figsize=(6*len(cases), 4))
if len(cases) == 1:
    axes = [axes]

for i, (gap, eps, label, ratio) in enumerate(cases):
    ax = axes[i]
    for mode, mode_label, color in [("std", "ERM", '#2196F3'), ("adv", "PGD-AT", '#FF5722')]:
        key = (gap, eps, mode)
        if key in all_grids:
            # Recompute boundary distances from the stored grid data
            gd, X_tr, y_tr = all_grids[key]
            from src.tessellation_analysis import estimate_boundary_distances_grid
            dists = estimate_boundary_distances_grid(
                gd["preds_grid"], X_tr, (-1.5, 1.5), RESOLUTION
            )
            ax.hist(dists, bins=30, alpha=0.5, label=mode_label,
                    color=color, density=True)

    ax.axvline(x=eps, color='red', linestyle='--', linewidth=2,
               label=f"ε={eps}", alpha=0.7)
    ax.set_xlabel("Distance to Decision Boundary")
    ax.set_ylabel("Density")
    ax.set_title(f"{label}\ngap={gap}, ε={eps}, ratio={ratio:.2f}")
    ax.legend(fontsize=8)

fig.suptitle("Boundary Distance Distributions Across Feasibility Regimes",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("/content/boundary_dists_regimes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved boundary_dists_regimes.png")

In [ ]:
# =====================================================
# FIGURE 6: Phase diagram — feasibility classification
# =====================================================
# Clean 2D phase diagram showing which (gap, eps) pairs are feasible
# with tessellation metric magnitude as background color

fig, ax = plt.subplots(figsize=(8, 6))

# Plot each (gap, eps) pair colored by regime
for _, row in merged.iterrows():
    feas = row["feasible_std"]
    gap_ratio = row["gap_ratio"]
    marker = 'o' if feas else 'X'
    color = '#2196F3' if feas else '#FF5722'
    # Size proportional to magnitude of tessellation reorganization
    size = abs(row["rel_diff_num_regions"]) * 200 + 30
    ax.scatter(row["eps"], row["gap"], c=color, s=size, marker=marker,
               edgecolors='black', linewidth=0.5, zorder=5, alpha=0.8)

# Draw feasibility boundary line: effective_gap = 2*eps
# Since effective_gap = gap / scale, and scale depends on gap,
# we compute it for a range of gaps
gap_range = np.linspace(0.05, 1.0, 100)
boundary_eps = []
for g in gap_range:
    _, _, eg = make_concentric_rings_gap(n_samples=100, gap=g, noise=NOISE, seed=SEED)
    boundary_eps.append(eg / 2)

ax.plot(boundary_eps, gap_range, 'r--', linewidth=2,
        label="Feasibility boundary\n(eff_gap = 2ε)")
ax.fill_betweenx(gap_range, boundary_eps, [max(EPS_VALUES)*1.1]*len(gap_range),
                  alpha=0.08, color='red', label="Infeasible region")

ax.set_xlabel("ε (perturbation budget)", fontsize=12)
ax.set_ylabel("Gap (pre-normalization)", fontsize=12)
ax.set_title("Phase Diagram: Feasibility of Robust Classification\n"
             "Size ∝ |tessellation reorganization|", fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/phase_diagram.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved phase_diagram.png")

## 11. Poisson Baseline Comparison

Compare the learned tessellation statistics to a matched-intensity Poisson hyperplane tessellation at selected (gap, ε) points.

In [ ]:
from src.tessellation_analysis import poisson_hyperplane_tessellation

# For a few representative cases, compare network tessellation to Poisson baseline
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, (gap, eps, label, ratio) in enumerate(cases):
    for row, mode in enumerate(["std", "adv"]):
        key = (gap, eps, mode)
        if key not in all_grids:
            continue
        # Get the network's boundary density to match
        r = [r for r in all_results
             if r["gap"] == gap and r["eps"] == eps
             and r["adv_enabled"] == (mode == "adv")]
        if not r:
            continue
        bd = r[0]["boundary_density"]
        nr = r[0]["num_regions"]

        # Generate matched Poisson baseline
        poisson = poisson_hyperplane_tessellation(
            intensity=bd, domain_range=(-1.0, 1.0),
            resolution=RESOLUTION, seed=42
        )

        ax = axes[row, col]
        categories = ["# Regions", "Boundary\nDensity"]
        network_vals = [nr, bd]
        poisson_vals = [poisson["num_regions"], poisson["boundary_density"]]

        x = np.arange(len(categories))
        width = 0.35
        ax.bar(x - width/2, network_vals, width, label="Network", color='#2196F3')
        ax.bar(x + width/2, poisson_vals, width, label="Poisson", color='#FF9800')
        ax.set_xticks(x)
        ax.set_xticklabels(categories)
        mode_label = "ERM" if mode == "std" else "PGD-AT"
        ax.set_title(f"{label} ({mode_label})\ngap={gap}, ε={eps}")
        ax.legend(fontsize=8)

axes[0, 0].set_ylabel("ERM", fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel("PGD-AT", fontsize=12, fontweight='bold')
fig.suptitle("Network vs. Poisson Baseline (matched boundary density)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("/content/poisson_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved poisson_comparison.png")

## 12. Save Results

In [ ]:
def serialize_value(v):
    """Convert numpy types to JSON-serializable Python types."""
    if isinstance(v, np.ndarray):
        return v.tolist()
    elif isinstance(v, (np.float32, np.float64)):
        return float(v)
    elif isinstance(v, (np.int32, np.int64)):
        return int(v)
    elif isinstance(v, (np.bool_,)):
        return bool(v)
    return v

results_out = {
    "experiment": "gap_epsilon_feasibility_sweep",
    "config": {
        "hidden_dims": HIDDEN_DIMS,
        "epochs": EPOCHS,
        "lr": LR,
        "scheduler": SCHEDULER,
        "pgd_steps": PGD_STEPS,
        "norm": NORM,
        "n_samples": N_SAMPLES,
        "noise": NOISE,
        "gap_values": GAP_VALUES,
        "eps_values": EPS_VALUES,
    },
    "results": [{k: serialize_value(v) for k, v in r.items()} for r in all_results],
}

os.makedirs("/content/results", exist_ok=True)
results_path = "/content/results/gap_eps_sweep_results.json"
with open(results_path, "w") as f:
    json.dump(results_out, f, indent=2)
print(f"Results saved to {results_path}")

# Also save the DataFrame as CSV
csv_path = "/content/results/gap_eps_sweep_summary.csv"
df.to_csv(csv_path, index=False)
print(f"Summary CSV saved to {csv_path}")

In [ ]:
# Download results (optional)
from google.colab import files

# Zip everything
!cd /content && zip -r gap_eps_sweep_results.zip results/ feasibility_*.png robust_acc_*.png phase_diagram.png regime_snapshots.png boundary_dists_regimes.png poisson_comparison.png 2>/dev/null

files.download("/content/gap_eps_sweep_results.zip")
print("Download ready.")

## 13. Interpretation Notes

**Expected findings in the feasible regime** (gap > 2ε):
- PGD-AT produces *fewer* linear regions than ERM (tessellation simplification)
- PGD-AT increases mean data-to-boundary distance (margins grow beyond ε)
- Local complexity near data decreases under PGD-AT
- Boundary density decreases (fewer, longer cell edges)

**Expected findings in the infeasible regime** (gap < 2ε):
- These differences diminish or reverse — PGD-AT cannot organize the tessellation for robustness because no robust classifier exists
- Robust accuracy collapses toward zero regardless of training
- Tessellation statistics between ERM and PGD-AT converge

**Connection to Shafahi et al. (2018):**
The isoperimetric bound is not just abstract — it manifests in the *geometry* of the tessellation. When robust classification is information-theoretically feasible, adversarial training reorganizes the linear regions to create wider margins. When it is infeasible, the tessellation cannot accommodate robustness, and the geometric signatures disappear.